# Multilingual Empathetic Dialogue System with Dynamic User-State Extraction

This project is a multilingual AI therapeutic assistant designed to support users who lack access to mental health services, offering empathy in English, Hindi, and Spanish (including code-switched phrases). At its core, the system uses Parameter-Efficient Fine-Tuning (PEFT/LoRA) on large models like Llama-3 or Qwen2.5 to provide nuanced emotional support. It also integrates Named Entity Recognition (NER) to extract user strengths, weaknesses, and goals, presenting them as interactive flashcards for self-reflection. Phase 1 is the crucial foundation where we set up the compute environment, clean and prepare our multilingual dataset, and verify that our tokenizer can handle complex language mixing. This ensures that the subsequent training and fine-tuning steps are built on a stable and high-quality data pipeline.

The three steps this notebook covers:
1. **Environment Setup**: Installing and verifying core dependencies.
2. **Dataset Preparation**: Loading and cleaning the EmpatheticDialogues dataset.
3. **Cross-Lingual Embeddings Pipeline**: Verifying multilingual tokenization capability.

## Section 1.1 — Compute Environment Setup

We begin by installing the core libraries required for our NLP pipeline. These include `transformers` for LLM management, `peft` for LoRA fine-tuning, `datasets` for data handling, `torch` for deep learning, `spacy` for NER tasks, and `flask`/`fastapi` to serve our eventual API endpoint.

In [ ]:
# Install all required libraries for the project
# Using -q for quiet installation to keep the notebook clean
!pip install -q transformers peft datasets torch spacy flask fastapi

After installation, we must verify that all libraries are correctly loaded and compatible. Printing the version numbers helps us confirm that the environment is ready for the deep learning tasks ahead.

In [ ]:
# Import libraries and check versions to confirm successful setup
import torch
import transformers
import peft
import datasets
import spacy
import flask
import fastapi

# Print version numbers for dependency tracking
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT version: {peft.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Spacy version: {spacy.__version__}")
print(f"Flask version: {flask.__version__}")
print(f"FastAPI version: {fastapi.__version__}")

## Section 1.2 — Dataset Preparation

We are using the `EmpatheticDialogues` dataset, which contains 25k conversations grounded in specific emotional situations. This dataset is ideal for training chatbots to respond with empathy, as it captures human-like emotional nuance across various contexts and intent categories.

In [ ]:
from datasets import load_dataset

# Load the EmpatheticDialogues dataset from Hugging Face
# This dataset serves as our primary training source for empathetic responses
raw_ds = load_dataset("empatheticdialogues")
print("Dataset structure successfully loaded:")
print(raw_ds)

To evaluate our model effectively, we split the data into training, validation, and testing sets. We aim for an 80/10/10 split to ensure we have enough data for training while keeping a representative portion for parameter tuning and final model performance evaluation.

In [ ]:
from datasets import concatenate_datasets

# Combine existing splits to perform a fresh, exact 80/10/10 split
full_ds = concatenate_datasets([raw_ds['train'], raw_ds['validation'], raw_ds['test']])

# Split 80% for training and 20% for testing/validation
train_testvalid = full_ds.train_test_split(test_size=0.2, seed=42)

# Split the 20% into two equal 10% parts for validation and testing
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

train_ds = train_testvalid['train']
valid_ds = test_valid['train']
test_ds = test_valid['test']

print(f"New splits - Train: {len(train_ds)}, Validation: {len(valid_ds)}, Test: {len(test_ds)}")

Conversational data often contains "noise" such as empty rows or redundant spaces that can skew model learning. We clean the dataset by removing null values, stripping unnecessary whitespace, and dropping exact duplicates to ensure high-quality training input.

In [ ]:
import pandas as pd

def clean_data(ds_obj, split_name):
    # Convert to pandas for easier cleaning operations
    df = ds_obj.to_pandas()
    initial_count = len(df)
    
    # Drop rows with null utterances and strip leading/trailing whitespace
    df = df.dropna(subset=['utterance'])
    df['utterance'] = df['utterance'].str.strip()
    
    # Remove exact duplicate utterances which add no new information
    df = df.drop_duplicates(subset=['utterance'])
    
    final_count = len(df)
    print(f"[{split_name}] Initial: {initial_count}, Cleaned: {final_count}, Removed: {initial_count - final_count}")
    return df

# Clean all three dataset splits and print statistics
train_clean = clean_data(train_ds, "Train")
valid_clean = clean_data(valid_ds, "Validation")
test_clean = clean_data(test_ds, "Test")

Noise in conversational data refers to artifacts like empty responses or repetitive text that provides no semantic value. By removing these, we prevent the model from learning biases from redundant data or crashing due to invalid input formats.

## Section 1.3 — Cross-Lingual Embeddings Pipeline

Code-switching is the practice of mixing multiple languages within a single conversation, a common trait in multilingual communities. Standard English tokenizers often fail to represent mixed-language sentences correctly, which is why we utilize XLM-RoBERTa for its superior multilingual vocabulary and cross-lingual understanding.

In [ ]:
from transformers import AutoTokenizer

# Load the XLM-RoBERTa tokenizer which supports over 100 languages
# We choose XLM-RoBERTa base as it offers better cross-lingual performance than mBERT
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
print(f"Successfully loaded tokenizer for: {tokenizer.name_or_path}")

We define a set of specific test sentences that blend English with Hindi and Spanish. These examples represent the natural, mixed-language style that our therapeutic assistant must be able to process to be effective in diverse linguistic environments.

In [ ]:
# Define exactly 6 test sentences featuring code-switching and a control sentence
test_sentences = [
    "I feel bahut anxious these days",                  # Hindi-English
    "Me siento very overwhelmed lately",              # Spanish-English
    "Mujhe lagta hai I am not good enough",          # Hindi-English
    "Estoy trying to be better every day",            # Spanish-English
    "I want to improve myself but kabhi kabhi it gets hard", # Hindi-English
    "I feel fine today"                               # Pure English (Control)
]
print(f"Prepared {len(test_sentences)} sentences for tokenization testing.")

Next, we tokenize each sentence to observe how the text is broken down into sub-word units and numerical IDs. This step verifies that the tokenizer correctly identifies and indexes words from all target languages without losing semantic information.

In [ ]:
# Tokenize all 6 sentences and display tokens along with their internal IDs
for i, text in enumerate(test_sentences, 1):
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    print(f"\nSentence {i}: {text}")
    print(f"Tokens: {tokens}")
    print(f"Token IDs: {token_ids}")

The tokenized output confirms that XLM-RoBERTa preserves the structure of code-switched sentences without defaulting to 'unknown' tokens for Hindi or Spanish words. This allows our therapeutic AI to accurately interpret the emotional state of users, regardless of language mixing.